<a href="https://colab.research.google.com/github/yilmajung/LLM_POC_Study_2025_v2/blob/main/r3_inference_backtest_homosex_abortion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Homosexual relationship

In [2]:
################
# CONFIG
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Base backbone
BASE_MODEL_NAME = "meta-llama/llama-3.1-8b"

# Experiments (point these to your trained folders from Exp-A/B/C)
ROOT = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_homosex_g4"
EXPERIMENTS = [
    {
        "name": "exp_A_train_le_2022",
        "SAVE_DIR": f"{ROOT}/exp_A_train_le_2022/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_A_train_le_2022/final_ckpt/two_head_exp_A_train_le_2022.pt",
        "FORECAST_YEARS": [2024],
    },
    {
        "name": "exp_B_train_le_2018",
        "SAVE_DIR": f"{ROOT}/exp_B_train_le_2018/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_B_train_le_2018/final_ckpt/two_head_exp_B_train_le_2018.pt",
        "FORECAST_YEARS": [2020, 2022, 2024],
    },
    {
        "name": "exp_C_train_le_2010",
        "SAVE_DIR": f"{ROOT}/exp_C_train_le_2010/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_C_train_le_2010/final_ckpt/two_head_exp_C_train_le_2010.pt",
        "FORECAST_YEARS": [2012, 2014, 2016, 2018, 2020, 2022, 2024],
    },
]

# Outcome toggle: 'homosex' | 'abortion'
OUTCOME = "homosex"

# Cross-sectional CSV (full, for evaluation)
CS_CSV = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/gss_abt_cs_homosex.csv"  # change per OUTCOME if separate files

# Grouping
GROUP_COLS = ["generation","gender","race","edu_level"]

################
# LABEL SPACES
ABORT4 = ["strong_anti", "anti", "pro", "strong_pro"]
HOMOSEX4 = ["always_wrong", "almost_always_wrong", "sometimes_wrong", "not_wrong_at_all"]

if OUTCOME == "homosex":
    CATS = HOMOSEX4
    TARGET_COL = "homosex"
elif OUTCOME == "abortion":
    CATS = ABORT4
    TARGET_COL = "abortion_att4"
else:
    raise ValueError("Unknown OUTCOME")

K = len(CATS)

################
# MODEL HEAD
class TwoHead(nn.Module):
    def __init__(self, hidden_size, K):
        super().__init__()
        self.head_row    = nn.Linear(hidden_size, K)   # Task A: transition row
        self.head_margin = nn.Linear(hidden_size, K)   # Task B: next margin
    def forward(self, feats):
        return self.head_row(feats), self.head_margin(feats)

def pooled_features(outputs, attention_mask, tail=96):
    hs = outputs.hidden_states[-1]   # [B,T,H]
    valid = attention_mask.sum(dim=1)
    feats = []
    for b in range(hs.size(0)):
        L = int(valid[b].item())
        s = max(0, L - tail); e = L
        if e <= s: s, e = max(0, L-32), L
        feats.append(hs[b, s:e, :].mean(dim=0))
    return torch.stack(feats, dim=0)

def _to_head_dtype(x, head_module):
    return x.to(head_module.head_row.weight.dtype)

################
# PROMPTED INFERENCE
@torch.no_grad()
def predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=None, max_len=768):
    if dt is None: dt = int(year_t1) - int(year_t)
    prompt = (
        "[Task: Predict transition row]\n"
        f"From: <Y{year_t}> → To: <Y{year_t1}> <DT{dt}>\n"
        f"Group: generation={group['generation']}; gender={group['gender']}; race={group['race']}; edu_level={group['edu_level']}\n"
        f"From option: {from_bin}\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    logits_row, _ = two_head(feats)
    p = F.softmax(logits_row, dim=1).float().cpu().numpy()[0]
    return p  # [K]

@torch.no_grad()
def predict_full_transition(model, two_head, tokenizer, group, year_t, year_t1, cats, max_len=768):
    T = np.zeros((len(cats), len(cats)), dtype=np.float32)
    dt = int(year_t1) - int(year_t)
    for from_bin in cats:
        p_row = predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=dt, max_len=max_len)
        i = cats.index(from_bin)
        T[i,:] = p_row
    T = np.clip(T, 1e-12, 1); T = T / T.sum(axis=1, keepdims=True)
    return T

@torch.no_grad()
def predict_next_margin(model, two_head, tokenizer, group, context, target_year, max_len=768):
    # context: list[(year, prob_vector)]
    ctx_parts = " ".join([f"<Y{yy}>[{','.join(f'{x:.4f}' for x in p)}]" for (yy,p) in context])
    prompt = (
        "[Task: Forecast next-wave margin]\n"
        f"Group: generation={group['generation']}; gender={group['gender']}; race={group['race']}; edu_level={group['edu_level']}\n"
        f"Context: {ctx_parts}\n"
        f"Predict: <Y{target_year}>\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    _, logits_margin = two_head(feats)
    p = F.softmax(logits_margin, dim=1).float().cpu().numpy()[0]
    return p

################
# OBSERVED MARGINS (FULL DATA, FOR EVAL ONLY)
cs = pd.read_csv(CS_CSV)
cs["year"] = cs["year"].apply(lambda x: 2020 if x==2021 else x)
cs = cs[cs[TARGET_COL].astype(str).str.strip().isin(CATS)].copy()
cs["wt"] = cs.get("wtssps", pd.Series([1.0]*len(cs)))
for c in GROUP_COLS: cs[c] = cs[c].astype(str).str.strip()

def weighted_probs(vals, wts, cats):
    d = {c:0.0 for c in cats}
    for v,w in zip(vals,wts):
        d[str(v).strip()] += float(w)
    vec = np.array([d[c] for c in cats], dtype=float)
    s = vec.sum()
    return vec/s if s>0 else None

p_cs_full = {}
for (gvals, df_g) in cs.groupby(GROUP_COLS):
    for y, df_y in df_g.groupby("year"):
        p = weighted_probs(df_y[TARGET_COL].tolist(), df_y["wt"].tolist(), CATS)
        if p is not None:
            p_cs_full[(gvals, int(y))] = p

################
# FORECAST DRIVER (per environment)
def jsd(p, q, eps=1e-9):
    p = np.clip(p, eps, 1); q = np.clip(q, eps, 1)
    p /= p.sum(); q /= q.sum(); m = 0.5*(p+q)
    return 0.5*np.sum(p*np.log(p/m)) + 0.5*np.sum(q*np.log(q/m))

def run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=None):
    """
    env: dict with SAVE_DIR, HEAD_PATH, FORECAST_YEARS
    Forecast rule: for each group g and each y* in FORECAST_YEARS, anchor on the nearest past observed year (y_prev < y*).
    Context includes y_prev and any y_prev-2, y_prev-4 that exist (never future).
    """
    SAVE_DIR = env["SAVE_DIR"]
    HEAD_PATH= env["HEAD_PATH"]
    TARGET_YEARS = env["FORECAST_YEARS"]

    # Load adapter + tokenizer + head for this environment
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, use_fast=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", output_hidden_states=True
    )
    model = PeftModel.from_pretrained(base, SAVE_DIR)
    model.eval()

    two_head = TwoHead(base.config.hidden_size, K).to(model.device)
    two_head.load_state_dict(torch.load(HEAD_PATH, map_location=model.device))
    two_head.eval()

    # Gather groups present in observed margins
    groups = sorted({g for (g, y) in p_cs_full.keys()})
    rows = []

    for g in groups:
        # find all observed years for this group
        ys = sorted(y for (gg,y) in p_cs_full.keys() if gg==g)
        if not ys: continue

        for y_star in TARGET_YEARS:
            # find nearest past observed year < y_star
            past = [y for y in ys if y < y_star]
            if not past:
                continue
            y_prev = max(past)
            p_prev = p_cs_full.get((g, y_prev), None)
            if p_prev is None:
                continue

            # build context (y_prev, y_prev-2, y_prev-4) if available
            ctx = []
            for L in context_lags:
                y_ctx = y_prev - L
                if (g, y_ctx) in p_cs_full:
                    ctx.append((y_ctx, p_cs_full[(g, y_ctx)]))
            if len(ctx)==0 and (g, y_prev) in p_cs_full:
                ctx.append((y_prev, p_cs_full[(g, y_prev)]))

            group = {"generation": g[0], "gender": g[1], "race": g[2], "edu_level": g[3]}

            # Transition forecast (match exact gap)
            T = predict_full_transition(model, two_head, tokenizer, group, y_prev, y_star, CATS)
            p_trans = p_prev @ T

            # Margin forecast (prompted)
            p_margin = predict_next_margin(model, two_head, tokenizer, group, ctx, y_star)

            # Ensemble
            p_hat = alpha * p_trans + (1 - alpha) * p_margin
            p_hat = np.clip(p_hat, 1e-12, 1); p_hat = p_hat / p_hat.sum()

            # Observed at target (for eval only)
            p_obs = p_cs_full.get((g, y_star), None)

            rec = {
                "experiment": env["name"],
                "generation": g[0], "gender": g[1], "race": g[2], "edu_level": g[3],
                "target_year": y_star,
                "anchor_year": y_prev,
            }
            for i,c in enumerate(CATS):
                rec[f"p_prev_{c}"]   = float(p_prev[i])
                rec[f"p_trans_{c}"]  = float(p_trans[i])
                rec[f"p_margin_{c}"] = float(p_margin[i])
                rec[f"p_hat_{c}"]    = float(p_hat[i])

            if p_obs is not None:
                rec.update({f"p_obs_{c}": float(p_obs[i]) for i,c in enumerate(CATS)})
                rec["JSD_hat_vs_obs"]   = float(jsd(p_hat, p_obs))
                rec["JSD_trans_vs_obs"] = float(jsd(p_trans, p_obs))
                rec["JSD_margin_vs_obs"]= float(jsd(p_margin, p_obs))

            rows.append(rec)

    df_env = pd.DataFrame(rows)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"forecasts_{OUTCOME}_{env['name']}.csv")
        df_env.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df_env

################
# RUN ALL ENVIRONMENTS
SAVE_RESULTS_DIR = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results"
all_results = []
for env in EXPERIMENTS:
    df_env = run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=SAVE_RESULTS_DIR)
    all_results.append(df_env)

df_all = pd.concat(all_results, ignore_index=True)
print("Combined shape:", df_all.shape)


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/forecasts_homosex_exp_A_train_le_2022.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/forecasts_homosex_exp_B_train_le_2018.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/forecasts_homosex_exp_C_train_le_2010.csv
Combined shape: (1301, 30)


In [3]:
# Abortion, Group 4

# Experiments (point these to your trained folders from Exp-A/B/C)
ROOT = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_abortion_g4"
EXPERIMENTS = [
    {
        "name": "exp_A_train_le_2022",
        "SAVE_DIR": f"{ROOT}/exp_A_train_le_2022/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_A_train_le_2022/final_ckpt/two_head_exp_A_train_le_2022.pt",
        "FORECAST_YEARS": [2024],
    },
    {
        "name": "exp_B_train_le_2018",
        "SAVE_DIR": f"{ROOT}/exp_B_train_le_2018/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_B_train_le_2018/final_ckpt/two_head_exp_B_train_le_2018.pt",
        "FORECAST_YEARS": [2020, 2022, 2024],
    },
    {
        "name": "exp_C_train_le_2010",
        "SAVE_DIR": f"{ROOT}/exp_C_train_le_2010/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_C_train_le_2010/final_ckpt/two_head_exp_C_train_le_2010.pt",
        "FORECAST_YEARS": [2012, 2014, 2016, 2018, 2020, 2022, 2024],
    },
]

# Outcome toggle: 'homosex' | 'abortion'
OUTCOME = "abortion"

# Cross-sectional CSV (full, for evaluation)
CS_CSV = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/gss_abt_cs_abortion.csv"  # change per OUTCOME if separate files

# Grouping
GROUP_COLS = ["generation","gender","race","edu_level"]

################
# LABEL SPACES
ABORT4 = ["strong_anti", "anti", "pro", "strong_pro"]
HOMOSEX4 = ["always_wrong", "almost_always_wrong", "sometimes_wrong", "not_wrong_at_all"]

if OUTCOME == "homosex":
    CATS = HOMOSEX4
    TARGET_COL = "homosex"
elif OUTCOME == "abortion":
    CATS = ABORT4
    TARGET_COL = "abortion_att4"
else:
    raise ValueError("Unknown OUTCOME")

K = len(CATS)

################
# MODEL HEAD
class TwoHead(nn.Module):
    def __init__(self, hidden_size, K):
        super().__init__()
        self.head_row    = nn.Linear(hidden_size, K)   # Task A: transition row
        self.head_margin = nn.Linear(hidden_size, K)   # Task B: next margin
    def forward(self, feats):
        return self.head_row(feats), self.head_margin(feats)

def pooled_features(outputs, attention_mask, tail=96):
    hs = outputs.hidden_states[-1]   # [B,T,H]
    valid = attention_mask.sum(dim=1)
    feats = []
    for b in range(hs.size(0)):
        L = int(valid[b].item())
        s = max(0, L - tail); e = L
        if e <= s: s, e = max(0, L-32), L
        feats.append(hs[b, s:e, :].mean(dim=0))
    return torch.stack(feats, dim=0)

def _to_head_dtype(x, head_module):
    return x.to(head_module.head_row.weight.dtype)

################
# PROMPTED INFERENCE
@torch.no_grad()
def predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=None, max_len=768):
    if dt is None: dt = int(year_t1) - int(year_t)
    prompt = (
        "[Task: Predict transition row]\n"
        f"From: <Y{year_t}> → To: <Y{year_t1}> <DT{dt}>\n"
        f"Group: generation={group['generation']}; gender={group['gender']}; race={group['race']}; edu_level={group['edu_level']}\n"
        f"From option: {from_bin}\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    logits_row, _ = two_head(feats)
    p = F.softmax(logits_row, dim=1).float().cpu().numpy()[0]
    return p  # [K]

@torch.no_grad()
def predict_full_transition(model, two_head, tokenizer, group, year_t, year_t1, cats, max_len=768):
    T = np.zeros((len(cats), len(cats)), dtype=np.float32)
    dt = int(year_t1) - int(year_t)
    for from_bin in cats:
        p_row = predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=dt, max_len=max_len)
        i = cats.index(from_bin)
        T[i,:] = p_row
    T = np.clip(T, 1e-12, 1); T = T / T.sum(axis=1, keepdims=True)
    return T

@torch.no_grad()
def predict_next_margin(model, two_head, tokenizer, group, context, target_year, max_len=768):
    # context: list[(year, prob_vector)]
    ctx_parts = " ".join([f"<Y{yy}>[{','.join(f'{x:.4f}' for x in p)}]" for (yy,p) in context])
    prompt = (
        "[Task: Forecast next-wave margin]\n"
        f"Group: generation={group['generation']}; gender={group['gender']}; race={group['race']}; edu_level={group['edu_level']}\n"
        f"Context: {ctx_parts}\n"
        f"Predict: <Y{target_year}>\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    _, logits_margin = two_head(feats)
    p = F.softmax(logits_margin, dim=1).float().cpu().numpy()[0]
    return p

################
# OBSERVED MARGINS (FULL DATA, FOR EVAL ONLY)
cs = pd.read_csv(CS_CSV)
cs["year"] = cs["year"].apply(lambda x: 2020 if x==2021 else x)
cs = cs[cs[TARGET_COL].astype(str).str.strip().isin(CATS)].copy()
cs["wt"] = cs.get("wtssps", pd.Series([1.0]*len(cs)))
for c in GROUP_COLS: cs[c] = cs[c].astype(str).str.strip()

def weighted_probs(vals, wts, cats):
    d = {c:0.0 for c in cats}
    for v,w in zip(vals,wts):
        d[str(v).strip()] += float(w)
    vec = np.array([d[c] for c in cats], dtype=float)
    s = vec.sum()
    return vec/s if s>0 else None

p_cs_full = {}
for (gvals, df_g) in cs.groupby(GROUP_COLS):
    for y, df_y in df_g.groupby("year"):
        p = weighted_probs(df_y[TARGET_COL].tolist(), df_y["wt"].tolist(), CATS)
        if p is not None:
            p_cs_full[(gvals, int(y))] = p

################
# FORECAST DRIVER (per environment)
def jsd(p, q, eps=1e-9):
    p = np.clip(p, eps, 1); q = np.clip(q, eps, 1)
    p /= p.sum(); q /= q.sum(); m = 0.5*(p+q)
    return 0.5*np.sum(p*np.log(p/m)) + 0.5*np.sum(q*np.log(q/m))

def run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=None):
    """
    env: dict with SAVE_DIR, HEAD_PATH, FORECAST_YEARS
    Forecast rule: for each group g and each y* in FORECAST_YEARS, anchor on the nearest past observed year (y_prev < y*).
    Context includes y_prev and any y_prev-2, y_prev-4 that exist (never future).
    """
    SAVE_DIR = env["SAVE_DIR"]
    HEAD_PATH= env["HEAD_PATH"]
    TARGET_YEARS = env["FORECAST_YEARS"]

    # Load adapter + tokenizer + head for this environment
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, use_fast=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", output_hidden_states=True
    )
    model = PeftModel.from_pretrained(base, SAVE_DIR)
    model.eval()

    two_head = TwoHead(base.config.hidden_size, K).to(model.device)
    two_head.load_state_dict(torch.load(HEAD_PATH, map_location=model.device))
    two_head.eval()

    # Gather groups present in observed margins
    groups = sorted({g for (g, y) in p_cs_full.keys()})
    rows = []

    for g in groups:
        # find all observed years for this group
        ys = sorted(y for (gg,y) in p_cs_full.keys() if gg==g)
        if not ys: continue

        for y_star in TARGET_YEARS:
            # find nearest past observed year < y_star
            past = [y for y in ys if y < y_star]
            if not past:
                continue
            y_prev = max(past)
            p_prev = p_cs_full.get((g, y_prev), None)
            if p_prev is None:
                continue

            # build context (y_prev, y_prev-2, y_prev-4) if available
            ctx = []
            for L in context_lags:
                y_ctx = y_prev - L
                if (g, y_ctx) in p_cs_full:
                    ctx.append((y_ctx, p_cs_full[(g, y_ctx)]))
            if len(ctx)==0 and (g, y_prev) in p_cs_full:
                ctx.append((y_prev, p_cs_full[(g, y_prev)]))

            group = {"generation": g[0], "gender": g[1], "race": g[2], "edu_level": g[3]}

            # Transition forecast (match exact gap)
            T = predict_full_transition(model, two_head, tokenizer, group, y_prev, y_star, CATS)
            p_trans = p_prev @ T

            # Margin forecast (prompted)
            p_margin = predict_next_margin(model, two_head, tokenizer, group, ctx, y_star)

            # Ensemble
            p_hat = alpha * p_trans + (1 - alpha) * p_margin
            p_hat = np.clip(p_hat, 1e-12, 1); p_hat = p_hat / p_hat.sum()

            # Observed at target (for eval only)
            p_obs = p_cs_full.get((g, y_star), None)

            rec = {
                "experiment": env["name"],
                "generation": g[0], "gender": g[1], "race": g[2], "edu_level": g[3],
                "target_year": y_star,
                "anchor_year": y_prev,
            }
            for i,c in enumerate(CATS):
                rec[f"p_prev_{c}"]   = float(p_prev[i])
                rec[f"p_trans_{c}"]  = float(p_trans[i])
                rec[f"p_margin_{c}"] = float(p_margin[i])
                rec[f"p_hat_{c}"]    = float(p_hat[i])

            if p_obs is not None:
                rec.update({f"p_obs_{c}": float(p_obs[i]) for i,c in enumerate(CATS)})
                rec["JSD_hat_vs_obs"]   = float(jsd(p_hat, p_obs))
                rec["JSD_trans_vs_obs"] = float(jsd(p_trans, p_obs))
                rec["JSD_margin_vs_obs"]= float(jsd(p_margin, p_obs))

            rows.append(rec)

    df_env = pd.DataFrame(rows)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"forecasts_{OUTCOME}_{env['name']}.csv")
        df_env.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df_env

################
# RUN ALL ENVIRONMENTS
SAVE_RESULTS_DIR = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results"
all_results = []
for env in EXPERIMENTS:
    df_env = run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=SAVE_RESULTS_DIR)
    all_results.append(df_env)

df_all = pd.concat(all_results, ignore_index=True)
print("Combined shape:", df_all.shape)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipython-input-559634288.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-559634288.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/forecasts_abortion_exp_A_train_le_2022.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/forecasts_abortion_exp_B_train_le_2018.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/forecasts_abortion_exp_C_train_le_2010.csv
Combined shape: (1167, 30)


In [4]:
# HOMOSEX, Group 1

# Experiments (point these to your trained folders from Exp-A/B/C)
ROOT = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_homosex_g1"
EXPERIMENTS = [
    {
        "name": "exp_A_train_le_2022",
        "SAVE_DIR": f"{ROOT}/exp_A_train_le_2022/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_A_train_le_2022/final_ckpt/two_head_exp_A_train_le_2022.pt",
        "FORECAST_YEARS": [2024],
    },
    {
        "name": "exp_B_train_le_2018",
        "SAVE_DIR": f"{ROOT}/exp_B_train_le_2018/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_B_train_le_2018/final_ckpt/two_head_exp_B_train_le_2018.pt",
        "FORECAST_YEARS": [2020, 2022, 2024],
    },
    {
        "name": "exp_C_train_le_2010",
        "SAVE_DIR": f"{ROOT}/exp_C_train_le_2010/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_C_train_le_2010/final_ckpt/two_head_exp_C_train_le_2010.pt",
        "FORECAST_YEARS": [2012, 2014, 2016, 2018, 2020, 2022, 2024],
    },
]

# Outcome toggle: 'homosex' | 'abortion'
OUTCOME = "homosex"

# Cross-sectional CSV (full, for evaluation)
CS_CSV = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/gss_abt_cs_homosex.csv"  # change per OUTCOME if separate files

# Grouping
GROUP_COLS = ["generation"]

################
# LABEL SPACES
ABORT4 = ["strong_anti", "anti", "pro", "strong_pro"]
HOMOSEX4 = ["always_wrong", "almost_always_wrong", "sometimes_wrong", "not_wrong_at_all"]

if OUTCOME == "homosex":
    CATS = HOMOSEX4
    TARGET_COL = "homosex"
elif OUTCOME == "abortion":
    CATS = ABORT4
    TARGET_COL = "abortion_att4"
else:
    raise ValueError("Unknown OUTCOME")

K = len(CATS)

################
# MODEL HEAD
class TwoHead(nn.Module):
    def __init__(self, hidden_size, K):
        super().__init__()
        self.head_row    = nn.Linear(hidden_size, K)   # Task A: transition row
        self.head_margin = nn.Linear(hidden_size, K)   # Task B: next margin
    def forward(self, feats):
        return self.head_row(feats), self.head_margin(feats)

def pooled_features(outputs, attention_mask, tail=96):
    hs = outputs.hidden_states[-1]   # [B,T,H]
    valid = attention_mask.sum(dim=1)
    feats = []
    for b in range(hs.size(0)):
        L = int(valid[b].item())
        s = max(0, L - tail); e = L
        if e <= s: s, e = max(0, L-32), L
        feats.append(hs[b, s:e, :].mean(dim=0))
    return torch.stack(feats, dim=0)

def _to_head_dtype(x, head_module):
    return x.to(head_module.head_row.weight.dtype)

################
# PROMPTED INFERENCE
@torch.no_grad()
def predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=None, max_len=768):
    if dt is None: dt = int(year_t1) - int(year_t)
    prompt = (
        "[Task: Predict transition row]\n"
        f"From: <Y{year_t}> → To: <Y{year_t1}> <DT{dt}>\n"
        f"Group: generation={group['generation']}\n"
        f"From option: {from_bin}\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    logits_row, _ = two_head(feats)
    p = F.softmax(logits_row, dim=1).float().cpu().numpy()[0]
    return p  # [K]

@torch.no_grad()
def predict_full_transition(model, two_head, tokenizer, group, year_t, year_t1, cats, max_len=768):
    T = np.zeros((len(cats), len(cats)), dtype=np.float32)
    dt = int(year_t1) - int(year_t)
    for from_bin in cats:
        p_row = predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=dt, max_len=max_len)
        i = cats.index(from_bin)
        T[i,:] = p_row
    T = np.clip(T, 1e-12, 1); T = T / T.sum(axis=1, keepdims=True)
    return T

@torch.no_grad()
def predict_next_margin(model, two_head, tokenizer, group, context, target_year, max_len=768):
    # context: list[(year, prob_vector)]
    ctx_parts = " ".join([f"<Y{yy}>[{','.join(f'{x:.4f}' for x in p)}]" for (yy,p) in context])
    prompt = (
        "[Task: Forecast next-wave margin]\n"
        f"Group: generation={group['generation']}\n"
        f"Context: {ctx_parts}\n"
        f"Predict: <Y{target_year}>\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    _, logits_margin = two_head(feats)
    p = F.softmax(logits_margin, dim=1).float().cpu().numpy()[0]
    return p

################
# OBSERVED MARGINS (FULL DATA, FOR EVAL ONLY)
cs = pd.read_csv(CS_CSV)
cs["year"] = cs["year"].apply(lambda x: 2020 if x==2021 else x)
cs = cs[cs[TARGET_COL].astype(str).str.strip().isin(CATS)].copy()
cs["wt"] = cs.get("wtssps", pd.Series([1.0]*len(cs)))
for c in GROUP_COLS: cs[c] = cs[c].astype(str).str.strip()

def weighted_probs(vals, wts, cats):
    d = {c:0.0 for c in cats}
    for v,w in zip(vals,wts):
        d[str(v).strip()] += float(w)
    vec = np.array([d[c] for c in cats], dtype=float)
    s = vec.sum()
    return vec/s if s>0 else None

p_cs_full = {}
for (gvals, df_g) in cs.groupby(GROUP_COLS):
    for y, df_y in df_g.groupby("year"):
        p = weighted_probs(df_y[TARGET_COL].tolist(), df_y["wt"].tolist(), CATS)
        if p is not None:
            p_cs_full[(gvals, int(y))] = p

################
# FORECAST DRIVER (per environment)
def jsd(p, q, eps=1e-9):
    p = np.clip(p, eps, 1); q = np.clip(q, eps, 1)
    p /= p.sum(); q /= q.sum(); m = 0.5*(p+q)
    return 0.5*np.sum(p*np.log(p/m)) + 0.5*np.sum(q*np.log(q/m))

def run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=None):
    """
    env: dict with SAVE_DIR, HEAD_PATH, FORECAST_YEARS
    Forecast rule: for each group g and each y* in FORECAST_YEARS, anchor on the nearest past observed year (y_prev < y*).
    Context includes y_prev and any y_prev-2, y_prev-4 that exist (never future).
    """
    SAVE_DIR = env["SAVE_DIR"]
    HEAD_PATH= env["HEAD_PATH"]
    TARGET_YEARS = env["FORECAST_YEARS"]

    # Load adapter + tokenizer + head for this environment
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, use_fast=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", output_hidden_states=True
    )
    model = PeftModel.from_pretrained(base, SAVE_DIR)
    model.eval()

    two_head = TwoHead(base.config.hidden_size, K).to(model.device)
    two_head.load_state_dict(torch.load(HEAD_PATH, map_location=model.device))
    two_head.eval()

    # Gather groups present in observed margins
    groups = sorted({g for (g, y) in p_cs_full.keys()})
    rows = []

    for g in groups:
        # find all observed years for this group
        ys = sorted(y for (gg,y) in p_cs_full.keys() if gg==g)
        if not ys: continue

        for y_star in TARGET_YEARS:
            # find nearest past observed year < y_star
            past = [y for y in ys if y < y_star]
            if not past:
                continue
            y_prev = max(past)
            p_prev = p_cs_full.get((g, y_prev), None)
            if p_prev is None:
                continue

            # build context (y_prev, y_prev-2, y_prev-4) if available
            ctx = []
            for L in context_lags:
                y_ctx = y_prev - L
                if (g, y_ctx) in p_cs_full:
                    ctx.append((y_ctx, p_cs_full[(g, y_ctx)]))
            if len(ctx)==0 and (g, y_prev) in p_cs_full:
                ctx.append((y_prev, p_cs_full[(g, y_prev)]))

            group = {"generation": g[0]}

            # Transition forecast (match exact gap)
            T = predict_full_transition(model, two_head, tokenizer, group, y_prev, y_star, CATS)
            p_trans = p_prev @ T

            # Margin forecast (prompted)
            p_margin = predict_next_margin(model, two_head, tokenizer, group, ctx, y_star)

            # Ensemble
            p_hat = alpha * p_trans + (1 - alpha) * p_margin
            p_hat = np.clip(p_hat, 1e-12, 1); p_hat = p_hat / p_hat.sum()

            # Observed at target (for eval only)
            p_obs = p_cs_full.get((g, y_star), None)

            rec = {
                "experiment": env["name"],
                "generation": g[0],
                "target_year": y_star,
                "anchor_year": y_prev,
            }
            for i,c in enumerate(CATS):
                rec[f"p_prev_{c}"]   = float(p_prev[i])
                rec[f"p_trans_{c}"]  = float(p_trans[i])
                rec[f"p_margin_{c}"] = float(p_margin[i])
                rec[f"p_hat_{c}"]    = float(p_hat[i])

            if p_obs is not None:
                rec.update({f"p_obs_{c}": float(p_obs[i]) for i,c in enumerate(CATS)})
                rec["JSD_hat_vs_obs"]   = float(jsd(p_hat, p_obs))
                rec["JSD_trans_vs_obs"] = float(jsd(p_trans, p_obs))
                rec["JSD_margin_vs_obs"]= float(jsd(p_margin, p_obs))

            rows.append(rec)

    df_env = pd.DataFrame(rows)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"forecasts_{OUTCOME}_{env['name']}.csv")
        df_env.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df_env

################
# RUN ALL ENVIRONMENTS
SAVE_RESULTS_DIR = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1"
all_results = []
for env in EXPERIMENTS:
    df_env = run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=SAVE_RESULTS_DIR)
    all_results.append(df_env)

df_all = pd.concat(all_results, ignore_index=True)
print("Combined shape:", df_all.shape)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipython-input-1847279268.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-1847279268.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1/forecasts_homosex_exp_A_train_le_2022.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1/forecasts_homosex_exp_B_train_le_2018.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1/forecasts_homosex_exp_C_train_le_2010.csv
Combined shape: (63, 27)


In [5]:
# ABORTION, Group 1

# Experiments (point these to your trained folders from Exp-A/B/C)
ROOT = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/outputs_gss_r_trial_abortion_g1"
EXPERIMENTS = [
    {
        "name": "exp_A_train_le_2022",
        "SAVE_DIR": f"{ROOT}/exp_A_train_le_2022/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_A_train_le_2022/final_ckpt/two_head_exp_A_train_le_2022.pt",
        "FORECAST_YEARS": [2024],
    },
    {
        "name": "exp_B_train_le_2018",
        "SAVE_DIR": f"{ROOT}/exp_B_train_le_2018/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_B_train_le_2018/final_ckpt/two_head_exp_B_train_le_2018.pt",
        "FORECAST_YEARS": [2020, 2022, 2024],
    },
    {
        "name": "exp_C_train_le_2010",
        "SAVE_DIR": f"{ROOT}/exp_C_train_le_2010/final_ckpt",
        "HEAD_PATH": f"{ROOT}/exp_C_train_le_2010/final_ckpt/two_head_exp_C_train_le_2010.pt",
        "FORECAST_YEARS": [2012, 2014, 2016, 2018, 2020, 2022, 2024],
    },
]

# Outcome toggle: 'homosex' | 'abortion'
OUTCOME = "abortion"

# Cross-sectional CSV (full, for evaluation)
CS_CSV = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/gss_abt_cs_abortion.csv"  # change per OUTCOME if separate files

# Grouping
GROUP_COLS = ["generation"]

################
# LABEL SPACES
ABORT4 = ["strong_anti", "anti", "pro", "strong_pro"]
HOMOSEX4 = ["always_wrong", "almost_always_wrong", "sometimes_wrong", "not_wrong_at_all"]

if OUTCOME == "homosex":
    CATS = HOMOSEX4
    TARGET_COL = "homosex"
elif OUTCOME == "abortion":
    CATS = ABORT4
    TARGET_COL = "abortion_att4"
else:
    raise ValueError("Unknown OUTCOME")

K = len(CATS)

################
# MODEL HEAD
class TwoHead(nn.Module):
    def __init__(self, hidden_size, K):
        super().__init__()
        self.head_row    = nn.Linear(hidden_size, K)   # Task A: transition row
        self.head_margin = nn.Linear(hidden_size, K)   # Task B: next margin
    def forward(self, feats):
        return self.head_row(feats), self.head_margin(feats)

def pooled_features(outputs, attention_mask, tail=96):
    hs = outputs.hidden_states[-1]   # [B,T,H]
    valid = attention_mask.sum(dim=1)
    feats = []
    for b in range(hs.size(0)):
        L = int(valid[b].item())
        s = max(0, L - tail); e = L
        if e <= s: s, e = max(0, L-32), L
        feats.append(hs[b, s:e, :].mean(dim=0))
    return torch.stack(feats, dim=0)

def _to_head_dtype(x, head_module):
    return x.to(head_module.head_row.weight.dtype)

################
# PROMPTED INFERENCE
@torch.no_grad()
def predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=None, max_len=768):
    if dt is None: dt = int(year_t1) - int(year_t)
    prompt = (
        "[Task: Predict transition row]\n"
        f"From: <Y{year_t}> → To: <Y{year_t1}> <DT{dt}>\n"
        f"Group: generation={group['generation']}\n"
        f"From option: {from_bin}\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    logits_row, _ = two_head(feats)
    p = F.softmax(logits_row, dim=1).float().cpu().numpy()[0]
    return p  # [K]

@torch.no_grad()
def predict_full_transition(model, two_head, tokenizer, group, year_t, year_t1, cats, max_len=768):
    T = np.zeros((len(cats), len(cats)), dtype=np.float32)
    dt = int(year_t1) - int(year_t)
    for from_bin in cats:
        p_row = predict_row_distribution(model, two_head, tokenizer, group, year_t, year_t1, from_bin, dt=dt, max_len=max_len)
        i = cats.index(from_bin)
        T[i,:] = p_row
    T = np.clip(T, 1e-12, 1); T = T / T.sum(axis=1, keepdims=True)
    return T

@torch.no_grad()
def predict_next_margin(model, two_head, tokenizer, group, context, target_year, max_len=768):
    # context: list[(year, prob_vector)]
    ctx_parts = " ".join([f"<Y{yy}>[{','.join(f'{x:.4f}' for x in p)}]" for (yy,p) in context])
    prompt = (
        "[Task: Forecast next-wave margin]\n"
        f"Group: generation={group['generation']}\n"
        f"Context: {ctx_parts}\n"
        f"Predict: <Y{target_year}>\n"
        "Answer:\n"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc["input_ids"].to(model.device); attn = enc["attention_mask"].to(model.device)
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    feats = pooled_features(out, attn, tail=96).to(model.device)
    feats = _to_head_dtype(feats, two_head)
    _, logits_margin = two_head(feats)
    p = F.softmax(logits_margin, dim=1).float().cpu().numpy()[0]
    return p

################
# OBSERVED MARGINS (FULL DATA, FOR EVAL ONLY)
cs = pd.read_csv(CS_CSV)
cs["year"] = cs["year"].apply(lambda x: 2020 if x==2021 else x)
cs = cs[cs[TARGET_COL].astype(str).str.strip().isin(CATS)].copy()
cs["wt"] = cs.get("wtssps", pd.Series([1.0]*len(cs)))
for c in GROUP_COLS: cs[c] = cs[c].astype(str).str.strip()

def weighted_probs(vals, wts, cats):
    d = {c:0.0 for c in cats}
    for v,w in zip(vals,wts):
        d[str(v).strip()] += float(w)
    vec = np.array([d[c] for c in cats], dtype=float)
    s = vec.sum()
    return vec/s if s>0 else None

p_cs_full = {}
for (gvals, df_g) in cs.groupby(GROUP_COLS):
    for y, df_y in df_g.groupby("year"):
        p = weighted_probs(df_y[TARGET_COL].tolist(), df_y["wt"].tolist(), CATS)
        if p is not None:
            p_cs_full[(gvals, int(y))] = p

################
# FORECAST DRIVER (per environment)
def jsd(p, q, eps=1e-9):
    p = np.clip(p, eps, 1); q = np.clip(q, eps, 1)
    p /= p.sum(); q /= q.sum(); m = 0.5*(p+q)
    return 0.5*np.sum(p*np.log(p/m)) + 0.5*np.sum(q*np.log(q/m))

def run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=None):
    """
    env: dict with SAVE_DIR, HEAD_PATH, FORECAST_YEARS
    Forecast rule: for each group g and each y* in FORECAST_YEARS, anchor on the nearest past observed year (y_prev < y*).
    Context includes y_prev and any y_prev-2, y_prev-4 that exist (never future).
    """
    SAVE_DIR = env["SAVE_DIR"]
    HEAD_PATH= env["HEAD_PATH"]
    TARGET_YEARS = env["FORECAST_YEARS"]

    # Load adapter + tokenizer + head for this environment
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, use_fast=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", output_hidden_states=True
    )
    model = PeftModel.from_pretrained(base, SAVE_DIR)
    model.eval()

    two_head = TwoHead(base.config.hidden_size, K).to(model.device)
    two_head.load_state_dict(torch.load(HEAD_PATH, map_location=model.device))
    two_head.eval()

    # Gather groups present in observed margins
    groups = sorted({g for (g, y) in p_cs_full.keys()})
    rows = []

    for g in groups:
        # find all observed years for this group
        ys = sorted(y for (gg,y) in p_cs_full.keys() if gg==g)
        if not ys: continue

        for y_star in TARGET_YEARS:
            # find nearest past observed year < y_star
            past = [y for y in ys if y < y_star]
            if not past:
                continue
            y_prev = max(past)
            p_prev = p_cs_full.get((g, y_prev), None)
            if p_prev is None:
                continue

            # build context (y_prev, y_prev-2, y_prev-4) if available
            ctx = []
            for L in context_lags:
                y_ctx = y_prev - L
                if (g, y_ctx) in p_cs_full:
                    ctx.append((y_ctx, p_cs_full[(g, y_ctx)]))
            if len(ctx)==0 and (g, y_prev) in p_cs_full:
                ctx.append((y_prev, p_cs_full[(g, y_prev)]))

            group = {"generation": g[0]}

            # Transition forecast (match exact gap)
            T = predict_full_transition(model, two_head, tokenizer, group, y_prev, y_star, CATS)
            p_trans = p_prev @ T

            # Margin forecast (prompted)
            p_margin = predict_next_margin(model, two_head, tokenizer, group, ctx, y_star)

            # Ensemble
            p_hat = alpha * p_trans + (1 - alpha) * p_margin
            p_hat = np.clip(p_hat, 1e-12, 1); p_hat = p_hat / p_hat.sum()

            # Observed at target (for eval only)
            p_obs = p_cs_full.get((g, y_star), None)

            rec = {
                "experiment": env["name"],
                "generation": g[0],
                "target_year": y_star,
                "anchor_year": y_prev,
            }
            for i,c in enumerate(CATS):
                rec[f"p_prev_{c}"]   = float(p_prev[i])
                rec[f"p_trans_{c}"]  = float(p_trans[i])
                rec[f"p_margin_{c}"] = float(p_margin[i])
                rec[f"p_hat_{c}"]    = float(p_hat[i])

            if p_obs is not None:
                rec.update({f"p_obs_{c}": float(p_obs[i]) for i,c in enumerate(CATS)})
                rec["JSD_hat_vs_obs"]   = float(jsd(p_hat, p_obs))
                rec["JSD_trans_vs_obs"] = float(jsd(p_trans, p_obs))
                rec["JSD_margin_vs_obs"]= float(jsd(p_margin, p_obs))

            rows.append(rec)

    df_env = pd.DataFrame(rows)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"forecasts_{OUTCOME}_{env['name']}.csv")
        df_env.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df_env

################
# RUN ALL ENVIRONMENTS
SAVE_RESULTS_DIR = "/content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1"
all_results = []
for env in EXPERIMENTS:
    df_env = run_environment(env, alpha=0.5, context_lags=(4,2,0), save_dir=SAVE_RESULTS_DIR)
    all_results.append(df_env)

df_all = pd.concat(all_results, ignore_index=True)
print("Combined shape:", df_all.shape)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipython-input-1483861176.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/tmp/ipython-input-1483861176.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1/forecasts_abortion_exp_A_train_le_2022.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1/forecasts_abortion_exp_B_train_le_2018.csv


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/LLM_POC_Study_2025_v2/r_trial_results/group1/forecasts_abortion_exp_C_train_le_2010.csv
Combined shape: (63, 27)
